## Data validation
- Validate data -> Save at dwh db of datawarehouse
    - Data Type not match Error -> set schema table at config file and prepare init table
    - Not null columns (key) -> set not null at config file
    - Key duplicate -> set key at config file
    - Row duplicate -> set row uniqueness validation in config file

In [1]:
import os
os.chdir("../")

In [2]:
os.getcwd()

'/Users/supawitjunsiritrakhoon/Desktop/Customer_Churn_Prediction/Project_file/customer-churn-prediction'

### Import library

In [3]:
import yaml
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Dict, List, Tuple, Any, Optional
from dataclasses import dataclass
import re
from datetime import datetime
from src.churn_prediction.logger import logger
from src.churn_prediction.pydantic.data_validation_config import DataValidationSchemaConfig
from src.churn_prediction.utils.common import load_single_config
from src.churn_prediction.utils.common import generate_sk_key

In [ ]:
@dataclass
class ValidationResult:
    """Stores validation results for a dataset."""
    is_valid: bool
    total_records: int
    valid_records: int
    invalid_records: int
    errors: List[Dict[str, Any]]
    warnings: List[Dict[str, Any]]
    quality_score: float
    
    def report(self) -> str:
        """Generate human-readable validation report."""
        report = f"""
{'='*70}
DATA VALIDATION REPORT
{'='*70}
Valid: {self.is_valid}
Quality Score: {self.quality_score:.2%}
Total Records: {self.total_records}
Valid Records: {self.valid_records}
Invalid Records: {self.invalid_records}

ERRORS ({len(self.errors)}):
{self._format_issues(self.errors)}

WARNINGS ({len(self.warnings)}):
{self._format_issues(self.warnings)}
{'='*70}
        """
        return report
    
    def _format_issues(self, issues: List[Dict]) -> str:
        """Format error/warning list."""
        if not issues:
            return "  None"
        return "\n".join([f"  - {issue['message']}" for issue in issues[:10]])


class DataValidator:
    """
    Validates data against schema configurations.
    
    Loads YAML schema files and applies comprehensive validation rules
    to pandas DataFrames including type checking, constraint validation,
    and data quality checks.
    """
    
    def __init__(self, schema_path: str):
        """
        Initialize validator with schema file.
        
        Args:
            schema_path (str): Path to schema YAML file
        """
        self.schema_path = Path(schema_path)
        self.schema_config = self._load_schema()
        self.columns_config = self.schema_config.get('columns', {})
        self.quality_rules = self.schema_config.get('quality_rules', {})

        logger.info(f"Schema loaded from {schema_path}")
    
    def _load_schema(self) -> Dict:
        """Load and parse YAML schema file."""
        try:
            with open(self.schema_path, 'r', encoding='utf-8') as f:
                schema = yaml.safe_load(f)
            schema_config = DataValidationSchemaConfig(**schema)
            logger.info(f"Successfully loaded schema: {self.schema_path}")
            return schema_config
        except FileNotFoundError:
            logger.error(f"Schema file not found: {self.schema_path}")
            raise
        except yaml.YAMLError as e:
            logger.error(f"Error parsing YAML schema: {e}")
            raise
    
    def validate(self, df: pd.DataFrame) -> ValidationResult:
        """
        Run all validation checks on DataFrame.
        
        Args:
            df (pd.DataFrame): Data to validate
        
        Returns:
            ValidationResult: Comprehensive validation results
        """
        errors = []
        warnings = []
        
        logger.info(f"Starting validation for {len(df)} records")
        
        # 1. Check required columns exist
        errors.extend(self._validate_columns_exist(df))
        
        # 2. Validate data types
        errors.extend(self._validate_data_types(df))
        
        # 3. Validate null/missing values
        warnings.extend(self._validate_nullability(df))
        
        # 4. Validate constraints (unique, foreign keys, etc.)
        errors.extend(self._validate_constraints(df))
        
        # 5. Validate business rules (enums, patterns, ranges)
        errors.extend(self._validate_business_rules(df))
        
        # 6. Detect anomalies
        warnings.extend(self._detect_anomalies(df))
        
        # Calculate quality score
        valid_records = len(df) - len([e for e in errors if 'row' in str(e)])
        quality_score = valid_records / len(df) if len(df) > 0 else 0
        
        is_valid = len(errors) == 0
        
        result = ValidationResult(
            is_valid=is_valid,
            total_records=len(df),
            valid_records=valid_records,
            invalid_records=len([e for e in errors if 'row' in str(e)]),
            errors=errors,
            warnings=warnings,
            quality_score=quality_score
        )
        
        logger.info(f"Validation complete. Quality Score: {quality_score:.2%}")
        return result
    
    def _validate_columns_exist(self, df: pd.DataFrame) -> List[Dict]:
        """Check all required columns exist in DataFrame."""
        errors = []
        required_columns = set(self.columns_config.keys())
        df_columns = set(df.columns)
        
        missing_columns = required_columns - df_columns
        if missing_columns:
            error = {
                'type': 'MissingColumn',
                'columns': list(missing_columns),
                'message': f"Missing required columns: {missing_columns}"
            }
            errors.append(error)
            logger.warning(f"Missing columns: {missing_columns}")
        
        extra_columns = df_columns - required_columns
        if extra_columns:
            logger.info(f"Extra columns found (will be ignored): {extra_columns}")
        
        return errors
    
    def _validate_data_types(self, df: pd.DataFrame) -> List[Dict]:
        """Validate column data types."""
        errors = []
        
        for column, config in self.columns_config.items():
            if column not in df.columns:
                continue
            
            expected_type = config.get('type')
            
            try:
                if expected_type == 'date':
                    # Try to parse as date
                    pd.to_datetime(df[column], errors='coerce')
                    null_count = df[column].isna().sum()
                    if null_count > 0 and not config.get('nullable', False):
                        errors.append({
                            'column': column,
                            'type': 'InvalidDateFormat',
                            'message': f"Column '{column}' has {null_count} invalid dates"
                        })
                
                elif expected_type == 'datetime':
                    pd.to_datetime(df[column], errors='coerce')
                    null_count = df[column].isna().sum()
                    if null_count > 0 and not config.get('nullable', False):
                        errors.append({
                            'column': column,
                            'type': 'InvalidDatetimeFormat',
                            'message': f"Column '{column}' has {null_count} invalid datetimes"
                        })
                
                elif expected_type == 'numeric':
                    pd.to_numeric(df[column], errors='coerce')
                    null_count = df[column].isna().sum()
                    if null_count > 0 and not config.get('nullable', False):
                        errors.append({
                            'column': column,
                            'type': 'InvalidNumeric',
                            'message': f"Column '{column}' has {null_count} non-numeric values"
                        })
            
            except Exception as e:
                logger.error(f"Error validating type for column '{column}': {e}")
        
        return errors
    
    def _validate_nullability(self, df: pd.DataFrame) -> List[Dict]:
        """Validate null/missing values against schema."""
        warnings = []
        null_tolerance = self.quality_rules.get('null_tolerance', {})
        
        for column, config in self.columns_config.items():
            if column not in df.columns:
                continue
            
            null_count = df[column].isna().sum()
            total_count = len(df)
            null_percentage = (null_count / total_count * 100) if total_count > 0 else 0
            
            is_nullable = config.get('nullable', False)
            tolerance = null_tolerance.get(column, 0)
            
            # Check if nulls exceed tolerance
            if null_percentage > tolerance:
                if not is_nullable and null_count > 0:
                    warnings.append({
                        'column': column,
                        'type': 'NullableViolation',
                        'null_percentage': null_percentage,
                        'message': f"Column '{column}' has {null_percentage:.2f}% null values "
                                   f"(tolerance: {tolerance}%)"
                    })
            
            logger.info(f"Column '{column}': {null_percentage:.2f}% null values")
        
        return warnings
    
    def _validate_constraints(self, df: pd.DataFrame) -> List[Dict]:
        """Validate primary key, unique, and foreign key constraints."""
        errors = []
        
        # Check unique constraints
        primary_keys = self.table_constraints.get('primary_keys', [])
        for column in primary_keys:
            if column not in df.columns:
                continue
            
            duplicate_count = df[column].duplicated().sum()
            if duplicate_count > 0:
                errors.append({
                    'column': column,
                    'type': 'DuplicateKeyViolation',
                    'duplicate_count': duplicate_count,
                    'message': f"Column '{column}' has {duplicate_count} duplicate values"
                })
                logger.error(f"Duplicate values found in '{column}'")
        
        # Check primary key
        primary_key = self.table_constraints.get('primary_key')
        if primary_key and primary_key in df.columns:
            duplicate_count = df[primary_key].duplicated().sum()
            null_count = df[primary_key].isna().sum()
            
            if duplicate_count > 0:
                errors.append({
                    'column': primary_key,
                    'type': 'PrimaryKeyDuplicate',
                    'duplicate_count': duplicate_count,
                    'message': f"Primary key '{primary_key}' has {duplicate_count} duplicates"
                })
            
            if null_count > 0:
                errors.append({
                    'column': primary_key,
                    'type': 'PrimaryKeyNull',
                    'null_count': null_count,
                    'message': f"Primary key '{primary_key}' has {null_count} null values"
                })
        
        return errors
    
    def _validate_business_rules(self, df: pd.DataFrame) -> List[Dict]:
        """Validate business rules (enums, patterns, ranges)."""
        errors = []
        
        for column, config in self.columns_config.items():
            if column not in df.columns:
                continue
            
            constraints = config.get('constraints', [])
            
            for constraint in constraints:
                constraint_type = constraint.get('type')
                
                # Enum constraint
                if constraint_type == 'enum':
                    allowed_values = constraint.get('values', [])
                    invalid_rows = ~df[column].isin(allowed_values) & df[column].notna()
                    invalid_count = invalid_rows.sum()
                    
                    if invalid_count > 0:
                        errors.append({
                            'column': column,
                            'type': 'EnumViolation',
                            'allowed_values': allowed_values,
                            'invalid_count': invalid_count,
                            'message': f"Column '{column}' has {invalid_count} values outside "
                                       f"allowed enum: {allowed_values}"
                        })
                        logger.warning(f"Enum violation in '{column}'")
                
                # Pattern constraint (regex)
                elif constraint_type == 'pattern':
                    pattern = constraint.get('value')
                    invalid_rows = df[column].astype(str).str.match(pattern) == False
                    invalid_count = invalid_rows.sum()
                    
                    if invalid_count > 0:
                        errors.append({
                            'column': column,
                            'type': 'PatternViolation',
                            'pattern': pattern,
                            'invalid_count': invalid_count,
                            'message': f"Column '{column}' has {invalid_count} values not matching "
                                       f"pattern: {pattern}"
                        })
                        logger.warning(f"Pattern violation in '{column}'")
                
                # Date range constraint
                elif constraint_type == 'date_range':
                    min_date = constraint.get('min')
                    max_date = constraint.get('max')
                    
                    try:
                        df_dates = pd.to_datetime(df[column], errors='coerce')
                        if min_date:
                            before_min = (df_dates < pd.to_datetime(min_date)).sum()
                            if before_min > 0:
                                errors.append({
                                    'column': column,
                                    'type': 'DateRangeViolation',
                                    'min_date': min_date,
                                    'violation_count': before_min,
                                    'message': f"Column '{column}' has {before_min} dates before {min_date}"
                                })
                        
                        if max_date:
                            after_max = (df_dates > pd.to_datetime(max_date)).sum()
                            if after_max > 0:
                                errors.append({
                                    'column': column,
                                    'type': 'DateRangeViolation',
                                    'max_date': max_date,
                                    'violation_count': after_max,
                                    'message': f"Column '{column}' has {after_max} dates after {max_date}"
                                })
                    except Exception as e:
                        logger.error(f"Error validating date range for '{column}': {e}")
        
        return errors
    
    def _detect_anomalies(self, df: pd.DataFrame) -> List[Dict]:
        """Detect statistical anomalies."""
        warnings = []
        statistical_bounds = self.quality_rules.get('statistical_bounds', {})
        
        if not statistical_bounds.get('enabled', False):
            return warnings
        
        for column, config in self.columns_config.items():
            if column not in df.columns:
                continue
            
            # Check for not_future constraint on dates
            constraints = config.get('constraints', [])
            for constraint in constraints:
                if constraint.get('type') == 'not_future':
                    try:
                        df_dates = pd.to_datetime(df[column], errors='coerce')
                        future_count = (df_dates > datetime.now()).sum()
                        
                        if future_count > 0:
                            warnings.append({
                                'column': column,
                                'type': 'FutureDate',
                                'anomaly_count': future_count,
                                'message': f"Column '{column}' has {future_count} future dates"
                            })
                            logger.warning(f"Future dates detected in '{column}'")
                    except Exception as e:
                        logger.error(f"Error checking future dates in '{column}': {e}")
        
        return warnings

In [47]:
# Example usage
if __name__ == "__main__":
    # Load schema
    schema_path = "src/churn_prediction/config/data_sources/schemas/customer_profile.schema.yaml"
    validator = DataValidator(schema_path)
    
    # Create sample data
    sample_data = {
        'user_id': ['1', '2', '3'],
        'first_name': ['John', 'Jane', 'Bob'],
        'last_name': ['Doe', 'Smith', 'Johnson'],
        'activated': ['True', 'True', 'False'],
        'admin_id': ['0', '0', '0'],
        'sex': ['male', 'female', 'male'],
        'foreigner': ['0', '0', '0'],
        'birthdate': ['1990-01-15', '1985-03-22', '1992-07-10'],
        'registed_time': ['2023-01-01 10:30:00', '2023-01-05 14:20:00', '2023-01-10 09:15:00']
    }
    
    df = pd.DataFrame(sample_data)
    
    # Validate data
    results = validator.validate(df)
    print(results.report())
    
    # Apply transformations
    df_clean = validator.apply_transformations(df)
    print("\nTransformed Data:")
    print(df_clean)

[ 2025-11-02 23:27:03 ] | churn_prediction | ERROR    | 1045903285.py:_load_schema:33 | Schema file not found: src/churn_prediction/config/data_sources/schemas/customer_profile.schema.yaml


FileNotFoundError: [Errno 2] No such file or directory: 'src/churn_prediction/config/data_sources/schemas/customer_profile.schema.yaml'

In [4]:
# def make_sample_data():
sample_data = {
    'user_id': ['1', '2', np.nan, '4', '4'],
    'first_name': ['John', 'Jane', 'Bob' ,'Alice', 'Alice'],
    'last_name': ['Doe', 'Smith', 'Johnson', 'Brown', 'Brown'],
    'activated': ['True', 'True', 'False', 'True', 'True'],
    'admin_id': ['0.1', '0', '0', '0', '0'],
    'sex': ['male', 'female', 'male', 'female', 'female'],
    'foreigner': ['0', '0', '0', '0', '0'],
    'birthdate': ['dfs', '1985-03-22', '1992-07-10', '1990-12-01', '1990-12-01'],
    'registed_time': ['2023-01-01 10:30:00', '2023-01-05 14:20:00', '2023-01-10 09:15:00', '2023-01-12 11:00:00', '2023-01-12 11:00:00']
}

df = pd.DataFrame(sample_data)

In [27]:
class DataValidator:
    """
    Validates data against schema configurations.
    
    Loads YAML schema files and applies comprehensive validation rules
    to pandas DataFrames including type checking, constraint validation,
    and data quality checks.
    """
    
    def __init__(self, schema_path: str):
        """
        Initialize validator with schema file.
        
        Args:
            schema_path (str): Path to schema YAML file
        """
        self.schema_path = Path(schema_path)
        self.schema_config = load_single_config(DataValidationSchemaConfig, self.schema_path)
        self.columns_config = self.schema_config.columns
        self.quality_rules_config = self.schema_config.quality_rules

        logger.info(f"Schema loaded from {schema_path}")
    

    def _validate_columns_exist(self, df: pd.DataFrame) -> List[Dict]:
        """Check all required columns exist in DataFrame."""
        errors = []
        required_columns = set(self.columns_config.keys())
        df_columns = set(df.columns)
        
        missing_columns = required_columns - df_columns
        if missing_columns:
            error_df = df.copy()
            error_df["error_type"] = "MissingColumn"
            error_df["error_message"] = f"Missing required columns: {missing_columns}"
            error = {
                'type': 'MissingColumn',
                'columns': list(missing_columns),
                'message': f"Missing required columns: {missing_columns}",
                'error_df': error_df
            }
            errors.append(error)
            logger.warning(f"Missing columns: {missing_columns}")
        
        extra_columns = df_columns - required_columns
        if extra_columns:
            logger.info(f"Extra columns found (will be ignored): {extra_columns}")
        
        return errors

    
    def _validate_data_types(self, df: pd.DataFrame) -> List[Dict]:
        """Validate column data types."""
        def _validate_datetime_column(df: pd.DataFrame, column: str, expected_type: str):
            check_df = df.copy()
            parsed_col = f"{column}_parsed"
            check_df[parsed_col] = pd.to_datetime(check_df[column], errors='coerce')

            error_mask = check_df[column].notna() & check_df[parsed_col].isna()
            if not error_mask.any():
                return None

            error_df = check_df[error_mask].drop(columns=[parsed_col])
            error_df["error_type"] = "InvalidDataType"
            error_df["error_message"] = f"Invalid column type '{expected_type}': {column}"

            return {
                'column': column,
                'type': f"Invalid{expected_type.capitalize()}Format",
                'message': f"Column '{column}' has {error_df.shape[0]} invalid {expected_type} values",
                'error_df': error_df
            }


        def _validate_numeric_column(df: pd.DataFrame, column: str, expected_type: str):
            check_df = df.copy()
            parsed_col = f"{column}_parsed"
            check_df[parsed_col] = pd.to_numeric(check_df[column], errors='coerce')

            error_mask = check_df[column].notna() & check_df[parsed_col].isna()
            if not error_mask.any():
                return None

            error_df = check_df[error_mask].drop(columns=[parsed_col])
            error_df["error_type"] = "InvalidDataType"
            error_df["error_message"] = f"Invalid column type 'numeric': {column}"

            return {
                'column': column,
                'type': "InvalidNumeric",
                'message': f"Column '{column}' has {error_df.shape[0]} non-numeric values",
                'error_df': error_df
            }


        def _validate_bool_column(df: pd.DataFrame, column: str, expected_type: str):
            check_df = df.copy()
            valid_values = {True, False, 'True', 'False', 1, 0}
            invalid_mask = ~check_df[column].isin(valid_values) & check_df[column].notna()

            if not invalid_mask.any():
                return None

            error_df = check_df[invalid_mask].copy()
            error_df["error_type"] = "InvalidDataType"
            error_df["error_message"] = f"Invalid column type 'bool': {column}"

            return {
                'column': column,
                'type': "InvalidBoolean",
                'message': f"Column '{column}' has {error_df.shape[0]} non-boolean values",
                'error_df': error_df
            }
        errors = []
        # Helper mapping between expected type and validator function
        type_validators = {
            'date': _validate_datetime_column,
            'datetime': _validate_datetime_column,
            'integer': _validate_numeric_column,
            'float': _validate_numeric_column,
            'bool': _validate_bool_column,
        }
        
        for column, config in self.columns_config.items():
            if column not in df.columns:
                continue
            
            expected_type = config.type.lower().strip()
            if expected_type == 'string':
                continue

            validator = type_validators.get(expected_type)

            if not validator:
                logger.warning(f"No validator defined for column type '{expected_type}' ({column})")
                continue

            try:
                error = validator(df, column, expected_type)
                if error:
                    errors.append(error)
            except Exception as e:
                logger.error(f"Error validating type for column '{column}': {e}")

        return errors
    
    def _validate_nullability(self, df: pd.DataFrame) -> List[Dict]:
        """Validate null/missing values against schema."""
        errors = []
        
        for column, config in self.columns_config.items():
            if column not in df.columns:
                continue

            is_nullable = config.nullable
            null_count = df[column].isna().sum()
            
            # Check if nulls 
            if not is_nullable and null_count > 0:
                error_df = df[df[column].isna()].copy()
                error_df["error_type"] = "NullableViolation"
                error_df["error_message"] = f"Column '{column}' has null values"
                errors.append({
                    'column': column,
                    'type': 'NullableViolation',
                    'message': f"Column '{column}' has null values",
                    'error_df': error_df
                })
            
            logger.info(f"Column '{column}': has null values")
        
        return errors

        
    def _validate_record_duplicates(self, df: pd.DataFrame):
        errors = []
        check_df = df.copy().drop("sk_key", axis=1, errors='ignore')
        duplicate_mask = check_df.duplicated(keep=False)
        error_df = df[duplicate_mask].copy()

        if not error_df.empty:
            error_df["error_type"] = "RecordDuplicateViolation"
            error_df["error_message"] = f"Duplicate records found based on all columns except"
            errors.append({
                'type': 'RecordDuplicateViolation',
                'message': f"Duplicate records found based on all columns except 'sk_key'",
                'error_df': error_df
            })
        return errors
        
    def _validate_unique_key_duplicates(self, df: pd.DataFrame):
        errors = []
        check_df = df.copy()
        key_columns = [column for column, config in self.columns_config.items() if config.primary_keys]
        duplicate_mask = check_df[key_columns].duplicated(keep=False)
        error_df = df[duplicate_mask].copy()

        if not error_df.empty:
            error_df["error_type"] = "UniqueKeyDuplicateViolation"
            error_df["error_message"] = f"Duplicate values found in unique key column: {key_columns}"
            errors.append({
                'type': 'UniqueKeyDuplicateViolation',
                'message': f"Duplicate values found in unique key column: {key_columns}",
                'error_df': error_df
            })
        return errors
                
        # allow_record_duplicates = self.quality_rules_config.allow_record_duplicates.enabled
        # if not allow_record_duplicates:

In [6]:
df = generate_sk_key(df)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   user_id        4 non-null      object
 1   first_name     5 non-null      object
 2   last_name      5 non-null      object
 3   activated      5 non-null      object
 4   admin_id       5 non-null      object
 5   sex            5 non-null      object
 6   foreigner      5 non-null      object
 7   birthdate      5 non-null      object
 8   registed_time  5 non-null      object
 9   sk_key         5 non-null      object
dtypes: object(10)
memory usage: 532.0+ bytes


In [28]:
schema_path = "src/churn_prediction/config/data_validation/customer_profile.yaml"
validator = DataValidator(schema_path)

In [34]:
validator.columns_config['first_name'].constraints

[ConstraintParameter(type='min_length', value=1, description=None),
 ConstraintParameter(type='max_length', value=100, description=None),
 ConstraintParameter(type='pattern', value="^[a-zA-Z\\s'-]+$", description='Letters, spaces, apostrophes, and hyphens only')]

In [23]:
df

,user_id,first_name,last_name,activated,admin_id,sex,foreigner,birthdate,registed_time,sk_key
0,1,John,Doe,True,0.1,male,0,dfs,2023-01-01 10:30:00,1
1,2,Jane,Smith,True,0,female,0,1985-03-22,2023-01-05 14:20:00,2
2,NaN,Bob,Johnson,False,0,male,0,1992-07-10,2023-01-10 09:15:00,3
3,4,Alice,Brown,True,0,female,0,1990-12-01,2023-01-12 11:00:00,4
4,4,Alice,Brown,True,0,female,0,1990-12-01,2023-01-12 11:00:00,5


In [17]:
key_columns = [column for column, config in validator.columns_config.items() if config.primary_keys]
duplicate_mask = df[key_columns].duplicated(keep=False)
duplicate_mask

0    False
1    False
2    False
3     True
4     True
dtype: bool